# 04 — SHAP Analysis

Visualize SHAP values for any scored product.
Verify the invariant: baseline + sum(SHAP_i) == VIBE_score

In [ ]:
import sys
sys.path.insert(0, '../')

from pathlib import Path
from pipeline.ingestion.ihut_extractor import extract_multiple_ihut_pdfs
from pipeline.features.base_extractor import Review
from pipeline.scoring.score_aggregator import compute_vibe_score
from pipeline.explainability.shap_explainer import explain
from pipeline.explainability.explanation_formatter import to_waterfall_data, format_key_drivers
import matplotlib.pyplot as plt
import numpy as np

# Load iHUT data
pdf_files = list(Path('../../').glob('*.pdf'))
all_docs = extract_multiple_ihut_pdfs(pdf_files)
reviews = [Review(text=c.text, source='ihut') for doc in all_docs for c in doc.verbatim_chunks]
all_chunks = [c for doc in all_docs for c in doc.chunks]

# Compute and explain
result = compute_vibe_score(reviews, all_chunks)
explanation = explain(result, product_id='chanel-iHUT')

print(f'VIBE Score: {explanation.vibe_score}')
print(f'Baseline:   {explanation.baseline_score}')
print(f'Sum SHAP:   {sum(c.shap_value for c in explanation.contributions):.2f}')
print(f'Invariant holds: {abs(explanation.baseline_score + sum(c.shap_value for c in explanation.contributions) - explanation.vibe_score) < 0.1}')

In [ ]:
# Waterfall visualization
waterfall = to_waterfall_data(explanation)
drivers = format_key_drivers(explanation)

labels = [e['label'] for e in waterfall]
values = [e['value'] for e in waterfall]
types = [e['type'] for e in waterfall]

colors = {'baseline': '#4a4a6a', 'positive': '#22c55e', 'negative': '#ef4444', 'total': '#8b5cf6', 'neutral': '#64748b'}

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#0f0f1a')
ax.set_facecolor('#0f0f1a')

cumulative = [e['cumulative'] for e in waterfall]

for i, (label, val, t) in enumerate(zip(labels, values, types)):
    color = colors.get(t, '#64748b')
    if t in ('baseline', 'total'):
        ax.barh(i, cumulative[i], color=color, alpha=0.8)
    else:
        start = cumulative[i] - val
        ax.barh(i, val, left=start, color=color, alpha=0.8)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, color='#aaaacc', fontsize=9)
ax.set_xlabel('VIBE Score Contribution', color='#8888aa')
ax.set_title(f'SHAP Waterfall — VIBE Score: {explanation.vibe_score}', color='white', fontsize=12)
ax.tick_params(colors='#8888aa')
plt.tight_layout()
plt.show()

In [ ]:
# Print plain-English drivers
print('\nNarrative:', explanation.narrative)
print('\nPositive drivers:')
for d in drivers['positive_drivers']:
    print(f'  {d["rank"]}. {d["dimension"]}: {d["impact"]} — {d["reason"]}')
print('\nLimiting factors:')
for d in drivers['negative_drivers']:
    print(f'  {d["rank"]}. {d["dimension"]}: {d["impact"]} — {d["reason"]}')